# 🤖 Agent Development - Risk Analyst & Compliance Officer

Welcome to Phase 2 & 3 of the project! In this notebook, you'll develop and test both AI agents:

1. **Risk Analyst Agent** (Chain-of-Thought prompting)
2. **Compliance Officer Agent** (ReACT prompting)

## 🎯 Learning Objectives
- Implement Chain-of-Thought prompting for systematic reasoning
- Build ReACT prompting for structured action-taking
- Handle structured JSON outputs from LLMs
- Create robust error handling and validation
- Test agents with real financial data

## 🚀 Setup and Environment

In [1]:
# Import required libraries
import os
import sys
import json
import openai
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime

# Project paths: src/ for the agent modules, the project root for the `src` package helpers
project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')
for path in (src_path, project_root):
    if path not in sys.path:
        sys.path.insert(0, path)

# Load environment variables
load_dotenv(os.path.join(project_root, '.env'))
MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

print("📚 Libraries loaded!")
print("🔐 Environment variables loaded")
print("📂 Source directory added to Python path:", src_path)
print(f"🤖 Model: {MODEL}")

📚 Libraries loaded!
🔐 Environment variables loaded
📂 Source directory added to Python path: /Users/maxlucaskienast/Desktop/Education/AI/Agentic_AI/Agentic_AI_Finance_Udacity/1_Prompting/cd14685-fin-serv-agentic-c1-classroom/project/starter/src
🤖 Model: gpt-4o-mini


In [2]:
# OpenAI Setup - Vocareum endpoint for voc- keys (Udacity workspace), standard OpenAI otherwise
from src import create_openai_client

client = create_openai_client()

✅ OpenAI client initialized (OpenAI key)
📍 Base URL: https://api.openai.com/v1 (default)


## 📊 Phase 1 Review: Load Foundation Components

Before building agents, let's ensure your foundation components are working.

In [3]:
# Import the implemented foundation components and both agents
from foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data
)
from risk_analyst_agent import RiskAnalystAgent, RISK_ANALYST_SYSTEM_PROMPT
from compliance_officer_agent import ComplianceOfficerAgent, COMPLIANCE_OFFICER_SYSTEM_PROMPT
from test_scenarios import RiskAnalystScenarios, ComplianceOfficerScenarios

print("✅ Foundation components imported")
print("✅ RiskAnalystAgent and ComplianceOfficerAgent imported")

✅ Foundation components imported
✅ RiskAnalystAgent and ComplianceOfficerAgent imported


In [4]:
# Build real cases for agent testing - one per typology found in Phase 1, plus a benign control
run_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
audit_log_path = os.path.join(project_root, 'outputs', 'audit_logs', f'agent_development_{run_stamp}.jsonl')
explainability_logger = ExplainabilityLogger(audit_log_path)
loader = DataLoader(explainability_logger)

customers_df, accounts_df, transactions_df = load_csv_data(os.path.join(project_root, 'data'))
account_records = accounts_df.to_dict('records')
transaction_records = transactions_df.to_dict('records')

# Customers identified by the targeted typology rules in 01_data_exploration.ipynb
evaluation_customers = {
    'CUST_0053': 'Structuring (7 cash deposits of $9.2k-$9.9k in 7 days)',
    'CUST_0062': 'Rapid movement (incoming $98k wire fanned out to 4 counterparties)',
    'CUST_0118': 'Cash velocity (cash/ATM activity across 6 branches in days)',
    'CUST_0125': 'Large cash (3 cash transactions of $22k-$45k)',
    'CUST_0002': 'Benign control (no rule triggered, low risk)'
}

real_cases = {}
for customer_id in evaluation_customers:
    customer_record = customers_df[customers_df['customer_id'] == customer_id].iloc[0].to_dict()
    real_cases[customer_id] = loader.create_case_from_data(customer_record, account_records, transaction_records)

print(f"📝 Audit log: {os.path.relpath(audit_log_path, project_root)}")
for customer_id, case in real_cases.items():
    print(f"📁 {customer_id} {case.customer.name:<18} {len(case.accounts)} account(s), "
          f"{len(case.transactions):>3} transactions - {evaluation_customers[customer_id]}")

📝 Audit log: outputs/audit_logs/agent_development_20261011_185411.jsonl
📁 CUST_0053 Tracy Lewis        3 account(s),  94 transactions - Structuring (7 cash deposits of $9.2k-$9.9k in 7 days)
📁 CUST_0062 Michael Stanley    3 account(s),  90 transactions - Rapid movement (incoming $98k wire fanned out to 4 counterparties)
📁 CUST_0118 Patrick Williams   3 account(s),  79 transactions - Cash velocity (cash/ATM activity across 6 branches in days)
📁 CUST_0125 Kayla Jackson      1 account(s),  14 transactions - Large cash (3 cash transactions of $22k-$45k)
📁 CUST_0002 Renee Blair        1 account(s),  11 transactions - Benign control (no rule triggered, low risk)


## 🔍 Phase 2: Risk Analyst Agent Development

The Risk Analyst Agent uses **Chain-of-Thought prompting** to systematically analyze suspicious activity patterns.

### 📚 Understanding Chain-of-Thought Prompting

Chain-of-Thought (CoT) prompting guides AI models through step-by-step reasoning:

1. **Explicit Steps**: Break complex reasoning into clear phases
2. **Sequential Logic**: Each step builds on previous ones
3. **Domain Expertise**: Frame AI as subject matter expert
4. **Structured Output**: Guide toward specific response format

In [5]:
# Inspect the Chain-of-Thought system prompt implemented in risk_analyst_agent.py
required_elements = {
    'Senior Financial Crime Risk Analyst persona': 'Senior Financial Crime Risk Analyst',
    'Regulatory context (CTR, structuring, SAR, OFAC)': '31 U.S.C. 5324',
    'Step 1 - Data Review': 'Step 1 - Data Review',
    'Step 2 - Pattern Recognition': 'Step 2 - Pattern Recognition',
    'Step 3 - Regulatory Mapping': 'Step 3 - Regulatory Mapping',
    'Step 4 - Risk Quantification': 'Step 4 - Risk Quantification',
    'Step 5 - Classification Decision': 'Step 5 - Classification Decision',
    'All 5 categories defined': 'Money_Laundering:',
    'Risk level definitions': 'RISK LEVELS',
    'Confidence calibration': 'CONFIDENCE SCORE',
    'JSON output contract': '"key_indicators"',
}

print("🧠 Chain-of-Thought Prompt Structure")
print("=" * 50)
for element, marker in required_elements.items():
    print(f"{'✅' if marker in RISK_ANALYST_SYSTEM_PROMPT else '❌'} {element}")
print(f"\n📏 Prompt length: {len(RISK_ANALYST_SYSTEM_PROMPT):,} characters (~{len(RISK_ANALYST_SYSTEM_PROMPT) // 4:,} tokens)")
print("\n" + "-" * 50)
print(RISK_ANALYST_SYSTEM_PROMPT)

🧠 Chain-of-Thought Prompt Structure
✅ Senior Financial Crime Risk Analyst persona
✅ Regulatory context (CTR, structuring, SAR, OFAC)
✅ Step 1 - Data Review
✅ Step 2 - Pattern Recognition
✅ Step 3 - Regulatory Mapping
✅ Step 4 - Risk Quantification
✅ Step 5 - Classification Decision
✅ All 5 categories defined
✅ Risk level definitions
✅ Confidence calibration
✅ JSON output contract

📏 Prompt length: 5,704 characters (~1,426 tokens)

--------------------------------------------------
You are a Senior Financial Crime Risk Analyst in a U.S. bank's Financial Intelligence Unit, with 15 years of experience in BSA/AML transaction monitoring, alert investigation and SAR decisioning. You analyze cases objectively, cite specific evidence (dates, amounts, transaction types, locations) and never speculate beyond the data provided.

TASK
Classify the suspicious activity in the case into exactly one category, quantify the risk, and explain your decision using the Chain-of-Thought framework below. Your

In [6]:
# Risk Analyst Agent - SIMPLE SMOKE TEST

def simple_risk_analyst_smoke_test():
    """Run one live analysis on a small synthetic structuring case and check the output structure"""
    print("🔍 Risk Analyst Smoke Test")
    try:
        agent = RiskAnalystAgent(client, explainability_logger, model=MODEL)
        test_case = RiskAnalystScenarios().create_case_data('structuring_classic')
        result = agent.analyze_case(test_case)

        assert isinstance(result, RiskAnalystOutput), "Result must be a RiskAnalystOutput"
        assert 0.0 <= result.confidence_score <= 1.0, "Confidence must be within 0-1"
        assert result.reasoning and result.key_indicators, "Reasoning and indicators must be populated"

        print(f"✅ SUCCESS: Got classification '{result.classification}' with confidence {result.confidence_score:.2f} "
              f"(risk level {result.risk_level})")
        print(f"🔑 Indicators: {result.key_indicators}")
        return result
    except Exception as e:
        print(f"❌ FAILED: {e}")
        raise

smoke_result = simple_risk_analyst_smoke_test()

🔍 Risk Analyst Smoke Test


✅ SUCCESS: Got classification 'Structuring' with confidence 0.90 (risk level High)
🔑 Indicators: ['$9,800.00 Cash_Deposit on 2024-01-15', '$9,500.00 Cash_Deposit on 2024-01-16', '$9,900.00 Cash_Deposit on 2024-01-17']


### 🧪 Risk Analyst Testing Framework

In [7]:
# COMPREHENSIVE Risk Analyst Testing - run the pre-built pytest suite from the project root
import inspect
import subprocess

tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

# Preview of the available tests
from test_risk_analyst import TestRiskAnalystAgent
test_methods = [name for name in dir(TestRiskAnalystAgent) if name.startswith('test_')]
print(f"📊 {len(test_methods)} Risk Analyst tests:")
for name in test_methods:
    print(f"   • {getattr(TestRiskAnalystAgent, name).__doc__}")

# Run in a subprocess so pytest's import handling matches the command line exactly
print("\n🚀 Running Risk Analyst test suite...")
pytest_run = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_risk_analyst.py', '-v', '-p', 'no:cacheprovider', '--color=no'],
    cwd=project_root, capture_output=True, text=True
)
print('\n'.join(line for line in pytest_run.stdout.splitlines() if '::' in line or 'passed' in line or 'failed' in line))
print("✅ All Risk Analyst tests passed!" if pytest_run.returncode == 0 else "❌ Some tests failed. Check the output above for details.")

📊 10 Risk Analyst tests:
   • Test RiskAnalystAgent initializes properly
   • Test handling of invalid JSON response
   • Test successful case analysis with valid response
   • Test OpenAI API call uses correct parameters
   • Test handling of empty LLM response
   • Test JSON extraction from code blocks
   • Test JSON extraction from plain text response
   • Test account formatting for prompts
   • Test transaction formatting for prompts
   • Test system prompt contains required elements

🚀 Running Risk Analyst test suite...


tests/test_risk_analyst.py::TestRiskAnalystAgent::test_agent_initialization PASSED [ 10%]
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_success PASSED [ 20%]
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_analyze_case_json_error PASSED [ 30%]
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_extract_json_from_code_block PASSED [ 40%]
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_extract_json_from_plain_text PASSED [ 50%]
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_extract_json_empty_response PASSED [ 60%]
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_format_accounts PASSED [ 70%]
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_format_transactions PASSED [ 80%]
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_system_prompt_structure PASSED [ 90%]
tests/test_risk_analyst.py::TestRiskAnalystAgent::test_api_call_parameters PASSED [100%]
============================== 10 passed in 0.26s ============================

### 🔬 Prompt Inspection: What the Agent Sees

`_format_case_for_prompt` turns a `CaseData` object into the user message:
- **Customer profile:** the SSN fragment and address are deliberately left out, because classification doesn't need them.
- **Accounts** and every **transaction**, in chronological order.
- **Pre-computed financial summary:** cash volume, near-threshold deposits, wire flows, activity on closed accounts, and description contradictions.

The summary does the arithmetic up front, so the model spends its reasoning on *interpretation* rather than adding up 94 transactions.

In [8]:
# Show the user prompt built for the structuring case
inspection_agent = RiskAnalystAgent(client, explainability_logger, model=MODEL)
user_prompt = inspection_agent._format_case_for_prompt(real_cases['CUST_0053'])
prompt_lines = user_prompt.splitlines()

print(f"📏 User prompt: {len(prompt_lines)} lines, {len(user_prompt):,} characters (~{len(user_prompt) // 4:,} tokens)")
print("=" * 70)
print('\n'.join(prompt_lines[:30]))
print(f"   ... {len(prompt_lines) - 38} more transactions ...")
print('\n'.join(prompt_lines[-8:]))

📏 User prompt: 121 lines, 10,941 characters (~2,735 tokens)
SUSPICIOUS ACTIVITY CASE cee6d0cb-d56e-4791-9095-f12cbbc3b038

CUSTOMER PROFILE
- Name: Tracy Lewis (CUST_0053)
- Age: 58 (born 1966-10-26)
- Occupation: Nurse, mental health
- Annual income: $131,925
- Customer since: 2024-05-11 (1 year)
- Internal risk rating: Medium

ACCOUNTS (3)
- CUST_0053_ACC_1 (Money_Market, Active): current balance $17,676.35, average monthly balance $89,530.85, opened 2024-07-23
- CUST_0053_ACC_2 (Savings, Active): current balance $62,101.19, average monthly balance $132,766.49, opened 2024-08-20
- CUST_0053_ACC_3 (Business_Checking, Active): current balance $90,679.35, average monthly balance $193,505.85, opened 2024-11-10

FINANCIAL SUMMARY
- Period: 2025-02-06 to 2025-07-31, 94 transactions, total volume $274,025.18
- Total volume is 2.1x reported annual income
- Volume by type: Cash_Deposit 7 txns $67,569.06; ACH_Credit 11 txns $30,509.50; ATM_Withdrawal 10 txns $29,375.45; ACH_Debit 10 txns $27,9

### 🧠 Chain-of-Thought in Action

The agent asks the model to write out all five steps *before* committing to a decision, and only then emit the JSON. The step-by-step trace is kept (`agent.last_reasoning_trace`) and written to the audit log next to the structured result, so an examiner can follow exactly how each classification was reached.

In [9]:
# Full Chain-of-Thought analysis of the real structuring case
risk_agent = RiskAnalystAgent(client, explainability_logger, model=MODEL)
structuring_result = risk_agent.analyze_case(real_cases['CUST_0053'])

print("🧠 REASONING TRACE")
print("=" * 70)
print(risk_agent.last_reasoning_trace)
print("\n📦 STRUCTURED OUTPUT (validated RiskAnalystOutput)")
print("=" * 70)
print(json.dumps(structuring_result.model_dump(), indent=2))

🧠 REASONING TRACE
CHAIN-OF-THOUGHT ANALYSIS  
Step 1 - Data Review: The customer, Tracy Lewis, is a nurse with an annual income of $131,925 and has been a customer for one year. She has three active accounts with significant balances. Over the review period, she conducted 94 transactions totaling $274,025.18, which is 2.1 times her annual income.  

Step 2 - Pattern Recognition: There are 7 cash deposits ranging from $9,238.15 to $9,915.61, totaling $67,569.06, all made within a week (2025-07-25 to 2025-07-31) across multiple branches. This pattern suggests an attempt to avoid the $10,000 CTR threshold.  

Step 3 - Regulatory Mapping: The cash deposits below the CTR threshold indicate structuring, as they are repeated cash amounts just under $10,000. Other categories such as sanctions, fraud, or money laundering do not apply based on the provided data.  

Step 4 - Risk Quantification: The total cash deposits are significant relative to her income, and the pattern of structuring is clea

### 📊 Real-Data Evaluation

The same agent run on the other four real cases: one per remaining typology from Phase 1, plus a benign control. The Phase 1 rules give a behavioural *expectation*. The agent has to reach its own conclusion from the raw case data.

In [10]:
# Analyze the remaining real cases and compare with the Phase 1 rule findings
real_results = {'CUST_0053': structuring_result}
for customer_id, case in real_cases.items():
    if customer_id not in real_results:
        real_results[customer_id] = risk_agent.analyze_case(case)

evaluation_table = pd.DataFrame([
    {
        'customer': f"{customer_id} {real_cases[customer_id].customer.name}",
        'phase_1_finding': evaluation_customers[customer_id].split(' (')[0],
        'classification': result.classification,
        'confidence': result.confidence_score,
        'risk_level': result.risk_level,
        'key_indicators': '; '.join(result.key_indicators[:3])
    }
    for customer_id, result in real_results.items()
])
print("📊 Real-Data Risk Analysis")
print("=" * 70)
print(evaluation_table.to_string(index=False))

print("\n💬 Condensed reasoning per case:")
for customer_id, result in real_results.items():
    print(f"\n🔸 {customer_id} ({result.classification}, {result.risk_level}):\n   {result.reasoning}")

📊 Real-Data Risk Analysis
                  customer phase_1_finding   classification  confidence risk_level                                                                                                                                                        key_indicators
     CUST_0053 Tracy Lewis     Structuring      Structuring        0.90       High                                                 7 cash deposits of $9,238.15-$9,915.61 across 5 branches in 7 days; Total cash deposits $67,569.06 just under $10,000
 CUST_0062 Michael Stanley  Rapid movement Money_Laundering        0.85       High            Incoming wire of $98,090.16 on 2025-07-27; Outgoing wires totaling $24,082.60 and $17,747.14 within 2 days; Total transaction volume is 3.0x annual income
CUST_0118 Patrick Williams   Cash velocity      Structuring        0.85       High Multiple cash deposits on 2025-08-02 totaling $23,404.96; 8 transactions posted to a closed account; 6 transactions with descriptions contradict

### 🧪 Synthetic Scenario Suite: All 5 Classification Types

The CSV extract has no sanctions exposure (Phase 1 finding). The pre-built scenarios in `src/test_scenarios.py` cover all five categories, including Sanctions, elder-fraud and a benign control, each with an expected classification, risk level and minimum confidence.

*`test_scenarios.py` was updated to match the implemented schemas. It previously used `date` instead of `transaction_date`, omitted the required `method` field, and built cases without `case_created_at`/`data_sources`.*

In [11]:
# Run all pre-built Risk Analyst scenarios
scenario_suite = RiskAnalystScenarios()
scenario_summary = scenario_suite.run_all_scenarios(risk_agent)

scenario_table = pd.DataFrame([
    {
        'scenario': name,
        'expected': result.get('expected_classification'),
        'actual': result.get('actual_classification'),
        'confidence': result.get('confidence_score'),
        'min_confidence': scenario_suite.scenarios[name]['min_confidence'],
        'expected_risk': result.get('expected_risk_level'),
        'actual_risk': result.get('actual_risk_level'),
        'pass': result.get('overall_pass')
    }
    for name, result in scenario_summary['detailed_results'].items()
])
print("\n" + scenario_table.to_string(index=False))

classification_accuracy = (scenario_table['expected'] == scenario_table['actual']).mean()
risk_level_accuracy = (scenario_table['expected_risk'] == scenario_table['actual_risk']).mean()
print(f"\n🎯 Classification accuracy: {classification_accuracy:.0%} | Risk level agreement: {risk_level_accuracy:.0%}")

🧪 Running Risk Analyst Scenario Tests...

🔍 Testing: Classic Structuring Pattern


   ✅ PASS - Classification: Structuring (confidence: 0.90)

🔍 Testing: Complex Money Laundering Pattern


   ✅ PASS - Classification: Money_Laundering (confidence: 0.90)

🔍 Testing: Identity Fraud Pattern


   ✅ PASS - Classification: Fraud (confidence: 0.85)

🔍 Testing: Sanctions Violation


   ✅ PASS - Classification: Sanctions (confidence: 1.00)

🔍 Testing: Benign Business Activity


   ✅ PASS - Classification: Other (confidence: 0.90)

📊 SUMMARY: 5/5 scenarios passed (100.0%)

                scenario         expected           actual  confidence  min_confidence expected_risk actual_risk  pass
     structuring_classic      Structuring      Structuring        0.90            0.80          High        High  True
money_laundering_complex Money_Laundering Money_Laundering        0.90            0.85      Critical        High  True
          fraud_identity            Fraud            Fraud        0.85            0.75          High        High  True
     sanctions_violation        Sanctions        Sanctions        1.00            0.90      Critical    Critical  True
         benign_activity            Other            Other        0.90            0.60           Low         Low  True

🎯 Classification accuracy: 100% | Risk level agreement: 80%


### 🛡️ Error Handling & Recovery

LLM output is untrusted input. The agent defends against each failure mode below. They are demonstrated with mocked API responses (no API cost) and logged to a separate demo audit file, so the real audit trail only holds real analyses:

| Failure mode | Handling |
|---|---|
| JSON wrapped in prose or code fences | Extractor searches fenced blocks, then any decodable object |
| Formatting slips (`"money laundering"`, `"85%"`, `"high"`, overlong reasoning, trailing commas) | Normalised or repaired, and every adjustment recorded in the audit entry |
| Sanctions classified below Critical | Policy floor raises the risk level to Critical (sanctions exposure is prohibited, not merely suspicious), and records the adjustment |
| No JSON / invalid schema on the first try | One **corrective retry** that feeds the exact error back to the model |
| Still invalid after retries | `ValueError`, with a failure audit entry |
| Truncated output (`finish_reason == "length"`) | Reported explicitly in the error |
| API failure (network, rate limit) | `RuntimeError`, with a failure audit entry |

In [12]:
# Demonstrate error handling with mocked LLM responses
import tempfile
from unittest.mock import Mock

def mock_response(content, finish_reason='stop'):
    response = Mock()
    response.choices = [Mock()]
    response.choices[0].message.content = content
    response.choices[0].finish_reason = finish_reason
    response.usage = None
    return response

valid_json = json.dumps({"classification": "Structuring", "confidence_score": 0.88,
                         "reasoning": "Profile: ... Pattern: ... Decision: Structuring",
                         "key_indicators": ["7 cash deposits $9.2k-$9.9k", "5 branches"], "risk_level": "High"})
sloppy_json = json.dumps({"classification": "money laundering", "confidence_score": "85%",
                          "reasoning": "Step-by-step analysis " * 40, "key_indicators": "pass-through wires",
                          "risk_level": "high"})
demo_scenarios = {
    'JSON in prose + code fence': [mock_response(f"Step 1 - Data Review: ...\n```json\n{valid_json}\n```\nDone.")],
    'Formatting slips normalised': [mock_response(sloppy_json)],
    'Trailing commas repaired': [mock_response(valid_json.replace('"High"}', '"High",}'))],
    'Corrective retry recovers': [mock_response("The activity looks like structuring."), mock_response(valid_json)],
    'Invalid category twice': [mock_response(valid_json.replace("Structuring", "Terrorism"))] * 2,
    'Truncated output twice': [mock_response('```json\n{"classification": "Struc', 'length')] * 2,
    'API connection failure': openai.APIConnectionError(request=Mock()),
}

demo_logger = ExplainabilityLogger(os.path.join(tempfile.mkdtemp(), 'error_handling_demo.jsonl'))
demo_case = real_cases['CUST_0053']
demo_rows = []
for label, responses in demo_scenarios.items():
    mock_client = Mock()
    mock_client.chat.completions.create.side_effect = responses
    demo_agent = RiskAnalystAgent(mock_client, demo_logger, model=MODEL)
    try:
        result = demo_agent.analyze_case(demo_case)
        outcome = f"✅ {result.classification} / {result.confidence_score} / {result.risk_level}"
    except (ValueError, RuntimeError) as e:
        outcome = f"🛑 {type(e).__name__}: {str(e).splitlines()[0][:70]}"
    entry = demo_logger.entries[-1]
    demo_rows.append({'scenario': label, 'api_calls': mock_client.chat.completions.create.call_count,
                      'outcome': outcome, 'audit_entry': f"success={entry['success']} | {entry['reasoning'][:45]}"})

print(pd.DataFrame(demo_rows).to_string(index=False))

# The adjustments the agent recorded in the audit entry for the sloppy response
_, adjustments = RiskAnalystAgent(Mock(), demo_logger)._normalize_output(json.loads(sloppy_json))
print("\n📝 Normalizations recorded for 'Formatting slips normalised':")
for adjustment in adjustments:
    print(f"   • {adjustment}")
assert len(demo_logger.entries) == len(demo_scenarios), "Exactly one audit entry per analysis"
print(f"\n✅ {len(demo_logger.entries)} analyses -> {len(demo_logger.entries)} audit entries (one per analysis, success or failure)")

                   scenario  api_calls                                                                              outcome                                                   audit_entry
 JSON in prose + code fence          1                                                          ✅ Structuring / 0.88 / High  success=True | Profile: ... Pattern: ... Decision: Structuri
Formatting slips normalised          1                                                     ✅ Money_Laundering / 0.85 / High  success=True | Step-by-step analysis Step-by-step analysis S
   Trailing commas repaired          1                                                          ✅ Structuring / 0.88 / High  success=True | Profile: ... Pattern: ... Decision: Structuri
  Corrective retry recovers          2                                                          ✅ Structuring / 0.88 / High  success=True | Profile: ... Pattern: ... Decision: Structuri
     Invalid category twice          2 🛑 ValueError: Risk Analyst outp

### 📝 Audit Trail & Cost Review

Every live analysis above was written to `outputs/audit_logs/`, together with the case-creation entries from the `DataLoader`. Each Risk Analyst entry includes the validated decision, the full Chain-of-Thought trace, any normalisations, the attempt count and token usage. That is the basis for the Phase 4 cost metrics.

In [13]:
# Review the Phase 2 audit trail and token usage
with open(audit_log_path) as f:
    audit_entries = [json.loads(line) for line in f]

analyst_entries = [e for e in audit_entries if e['agent_type'] == 'RiskAnalyst']
print("📝 Audit Trail Summary")
print("=" * 50)
print(f"📄 File: {os.path.relpath(audit_log_path, project_root)}")
print(f"🧾 Entries: {len(audit_entries)} total - "
      f"{sum(e['agent_type'] == 'DataLoader' for e in audit_entries)} DataLoader, {len(analyst_entries)} RiskAnalyst")
print(f"✅ Successful analyses: {sum(e['success'] for e in analyst_entries)} / {len(analyst_entries)}")
print(f"⏱️  Avg analysis latency: {sum(e['execution_time_ms'] for e in analyst_entries) / len(analyst_entries) / 1000:.1f} s")

# Token usage straight from the audit trail - each RiskAnalyst entry records its own usage and attempts
usage = {'api_calls': 0, 'prompt_tokens': 0, 'completion_tokens': 0}
for entry in analyst_entries:
    details = entry['output_summary']  # stored as a JSON object
    usage['api_calls'] += details['attempts']
    usage['prompt_tokens'] += details['token_usage']['prompt_tokens']
    usage['completion_tokens'] += details['token_usage']['completion_tokens']

# gpt-4o-mini list prices in USD per 1M tokens - update if the model or pricing changes
PRICE_PER_1M = {'prompt_tokens': 0.15, 'completion_tokens': 0.60}
estimated_cost = sum(usage[key] / 1_000_000 * price for key, price in PRICE_PER_1M.items())
print(f"\n💰 Token usage ({usage['api_calls']} API calls for this run): "
      f"{usage['prompt_tokens']:,} prompt + {usage['completion_tokens']:,} completion tokens")
print(f"💵 Estimated cost: ${estimated_cost:.4f} (~${estimated_cost / max(usage['api_calls'], 1):.5f} per analysis)")

print("\n🔍 Sample RiskAnalyst audit entry:")
sample_entry = dict(analyst_entries[0])
sample_entry['output_summary'] = {k: (v[:300] + '...' if isinstance(v, str) and len(v) > 300 else v) for k, v in sample_entry['output_summary'].items()}  # shorten long text fields for display
print(json.dumps(sample_entry, indent=2))

📝 Audit Trail Summary
📄 File: outputs/audit_logs/agent_development_20261011_185411.jsonl
🧾 Entries: 16 total - 5 DataLoader, 11 RiskAnalyst
✅ Successful analyses: 11 / 11
⏱️  Avg analysis latency: 4.1 s

💰 Token usage (11 API calls for this run): 29,357 prompt + 4,778 completion tokens
💵 Estimated cost: $0.0073 (~$0.00066 per analysis)

🔍 Sample RiskAnalyst audit entry:
{
  "timestamp": "2026-10-11T17:54:16.972815+00:00",
  "case_id": "CASE_STRUCTURING_CLASSIC",
  "agent_type": "RiskAnalyst",
  "action": "analyze_case",
  "input_summary": {
    "customer_id": "CUST_STRUCT_001",
    "num_accounts": 1,
    "num_transactions": 3,
    "model": "gpt-4o-mini"
  },
  "output_summary": {
    "classification": "Structuring",
    "confidence_score": 0.9,
    "reasoning": "Profile: Customer has a medium risk rating with no reported income. Pattern: Three cash deposits of $9,800, $9,500, and $9,900 over three days indicate structuring. Mapping: Evidence supports Structuring, while other categories

## ✅ Phase 3: Compliance Officer Agent Development

The Compliance Officer Agent uses **ReACT prompting** to generate regulatory-compliant SAR narratives.

### 📚 Understanding ReACT Prompting

ReACT (Reasoning + Action) prompting separates thinking and doing:

1. **Reasoning Phase**: Analyze situation and plan approach
2. **Action Phase**: Execute specific task with informed decisions
3. **Structured Workflow**: Consistent approach to complex tasks
4. **Regulatory Compliance**: Emphasis on meeting specific requirements

In [14]:
# Inspect the ReACT system prompt implemented in compliance_officer_agent.py
required_elements = {
    'Senior BSA/AML Compliance Officer persona': 'Senior BSA/AML Compliance Officer',
    'REASONING PHASE: THOUGHT 1 - Review findings': 'THOUGHT 1 - Review findings',
    'REASONING PHASE: THOUGHT 2 - Regulatory requirements': 'THOUGHT 2 - Regulatory requirements',
    'REASONING PHASE: THOUGHT 3 - Narrative plan (5 Ws)': 'THOUGHT 3 - Narrative plan',
    'ACTION PHASE: ACTION 1 - Draft': 'ACTION 1 - Draft',
    'ACTION PHASE: ACTION 2 - Verify with OBSERVATION': 'ACTION 2 - Verify',
    'ACTION PHASE: ACTION 3 - Finalize as JSON': 'ACTION 3 - Finalize',
    '120-word limit': 'at most 120 words',
    'WHO / WHAT / WHEN / WHERE / WHY coverage': 'WHO, WHAT, WHEN, WHERE, WHY',
    'Objective tone, no speculation': 'never assert guilt',
    'Approved regulatory citation list': 'REGULATORY CITATIONS (cite only from this list)',
    'JSON output contract': '"completeness_check"',
}

print("⚡ ReACT Prompt Structure")
print("=" * 50)
for element, marker in required_elements.items():
    print(f"{'✅' if marker in COMPLIANCE_OFFICER_SYSTEM_PROMPT else '❌'} {element}")
print(f"\n📏 Prompt length: {len(COMPLIANCE_OFFICER_SYSTEM_PROMPT):,} characters "
      f"(~{len(COMPLIANCE_OFFICER_SYSTEM_PROMPT) // 4:,} tokens)")
print("\n" + "-" * 50)
print(COMPLIANCE_OFFICER_SYSTEM_PROMPT)

⚡ ReACT Prompt Structure
✅ Senior BSA/AML Compliance Officer persona
✅ REASONING PHASE: THOUGHT 1 - Review findings
✅ REASONING PHASE: THOUGHT 2 - Regulatory requirements
✅ REASONING PHASE: THOUGHT 3 - Narrative plan (5 Ws)
✅ ACTION PHASE: ACTION 1 - Draft
✅ ACTION PHASE: ACTION 2 - Verify with OBSERVATION
✅ ACTION PHASE: ACTION 3 - Finalize as JSON
✅ 120-word limit
✅ WHO / WHAT / WHEN / WHERE / WHY coverage
✅ Objective tone, no speculation
✅ Approved regulatory citation list
✅ JSON output contract

📏 Prompt length: 4,635 characters (~1,158 tokens)

--------------------------------------------------
You are a Senior BSA/AML Compliance Officer at a U.S. bank, with 15 years of experience writing and quality-reviewing Suspicious Activity Report (SAR) narratives for FinCEN. You turn analyst findings into clear, factual, examiner-ready narratives.

TASK
Write the SAR narrative for the case, based on the Risk Analyst's findings and the case facts provided. The narrative is filed with FinCEN,

In [15]:
# Compliance Officer Agent - SIMPLE SMOKE TEST

def simple_compliance_officer_smoke_test():
    """Generate one live narrative from the Phase 2 smoke-test analysis and check its structure"""
    print("✅ Compliance Officer Smoke Test")
    try:
        agent = ComplianceOfficerAgent(client, explainability_logger, model=MODEL)
        test_case = RiskAnalystScenarios().create_case_data('structuring_classic')
        result = agent.generate_compliance_narrative(test_case, smoke_result)  # Risk Analyst output from Phase 2

        word_count = len(result.narrative.split())
        assert isinstance(result, ComplianceOfficerOutput), "Result must be a ComplianceOfficerOutput"
        assert word_count <= 120, f"Narrative has {word_count} words"
        assert result.regulatory_citations, "Narrative must cite regulations"

        print(f"✅ SUCCESS: Generated {word_count} word narrative (complete: {result.completeness_check})")
        print(f"Preview: {result.narrative[:160]}...")
        print(f"📚 Citations: {result.regulatory_citations}")
        return result
    except Exception as e:
        print(f"❌ FAILED: {e}")
        raise

compliance_smoke_result = simple_compliance_officer_smoke_test()

✅ Compliance Officer Smoke Test


✅ SUCCESS: Generated 68 word narrative (complete: True)
Preview: Sarah Thompson (CUST_STRUCT_001) made three cash deposits totaling $29,200.00 into account ACC_CHK_4567 between 2024-01-15 and 2024-01-17. The deposits were $9,...
📚 Citations: ['31 CFR 1020.320 (SAR filing requirement)', '31 USC 5324 (Structuring)']


### 🧪 Compliance Officer Testing Framework

In [16]:
# COMPREHENSIVE Compliance Officer Testing - run the pre-built pytest suite from the project root
from test_compliance_officer import TestComplianceOfficerAgent

test_methods = [name for name in dir(TestComplianceOfficerAgent) if name.startswith('test_')]
print(f"📝 {len(test_methods)} Compliance Officer tests:")
for name in test_methods:
    print(f"   • {getattr(TestComplianceOfficerAgent, name).__doc__}")

print("\n🚀 Running Compliance Officer test suite...")
pytest_run = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_compliance_officer.py', '-v', '-p', 'no:cacheprovider', '--color=no'],
    cwd=project_root, capture_output=True, text=True
)
print('\n'.join(line for line in pytest_run.stdout.splitlines() if '::' in line or 'passed' in line or 'failed' in line))
print("✅ All Compliance Officer tests passed!" if pytest_run.returncode == 0 else "❌ Some tests failed. Check the output above for details.")

📝 10 Compliance Officer tests:
   • Test ComplianceOfficerAgent initializes properly
   • Test OpenAI API call uses correct parameters
   • Test handling of empty LLM response
   • Test JSON extraction from code blocks
   • Test JSON extraction from plain text response
   • Test transaction formatting for compliance narratives
   • Test successful narrative generation with valid response
   • Test handling of invalid JSON response
   • Test narrative word count validation (120 word limit)
   • Test system prompt contains required ReACT elements

🚀 Running Compliance Officer test suite...


tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_agent_initialization PASSED [ 10%]
tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_generate_compliance_narrative_success PASSED [ 20%]
tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_narrative_word_count_validation PASSED [ 30%]
tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_json_parsing_error PASSED [ 40%]
tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_extract_json_from_code_block PASSED [ 50%]
tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_extract_json_from_plain_text PASSED [ 60%]
tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_extract_json_empty_response PASSED [ 70%]
tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_format_transactions_for_compliance PASSED [ 80%]
tests/test_compliance_officer.py::TestComplianceOfficerAgent::test_system_prompt_structure PASSED [ 90%]
tests/test_compl

### ⚡ ReACT in Action: Risk Analyst → Compliance Officer

The Phase 2 Chain-of-Thought analysis of the real structuring case (`structuring_result`) is handed to the Compliance Officer together with the case facts. The output shows both phases:
- **REASONING (THOUGHT 1-3):** what the analyst found, which regulations apply, and the WHO/WHAT/WHEN/WHERE/WHY plan.
- **ACTION (1-3):** draft, self-check with an OBSERVATION, final JSON.

The model's self-reported word count is **not trusted**. The agent validates every narrative in code (word limit, the five elements, citations, no SSN), and that validation is the authoritative observation. It also raises **reviewer warnings** for accuracy and tone: dollar amounts that can't be traced to the case facts (hallucinated figures), typologies other than the analyst's classification, and speculative language about intent or the source of funds.

In [17]:
# Generate the SAR narrative for the real structuring case from its Phase 2 risk analysis
import re

compliance_agent = ComplianceOfficerAgent(client, explainability_logger, model=MODEL)
structuring_case = real_cases['CUST_0053']
structuring_narrative = compliance_agent.generate_compliance_narrative(structuring_case, structuring_result)

print("⚡ ReACT TRACE (Reasoning + Action phases)")
print("=" * 70)
print(compliance_agent.last_react_trace)

validation = compliance_agent.last_validation
self_reported = re.search(r"OBSERVATION:\s*(\d+)\s*words", compliance_agent.last_react_trace)
print("\n📄 FINAL SAR NARRATIVE")
print("=" * 70)
print(structuring_narrative.narrative)
print(f"\n📏 Word count: {validation['word_count']} / 120 (model's self-reported OBSERVATION: "
      f"{self_reported.group(1) if self_reported else 'n/a'})")
print(f"🧩 Elements: {', '.join(f'{k.upper()} {chr(9989) if v else chr(10060)}' for k, v in validation['elements'].items())}")
print(f"📚 Citations: {structuring_narrative.regulatory_citations}")
print(f"🏷️  BSA/AML terminology: {validation['regulatory_terms']}")
print(f"✅ Completeness check: {structuring_narrative.completeness_check} | issues: {validation['issues'] or 'none'} | "
      f"warnings: {validation['warnings'] or 'none'}")
print(f"\n💭 Narrative reasoning: {structuring_narrative.narrative_reasoning}")

⚡ ReACT TRACE (Reasoning + Action phases)
THOUGHT 1: The Risk Analyst concluded that Tracy Lewis (CUST_0053) engaged in structuring, evidenced by 7 cash deposits ranging from $9,238.15 to $9,915.61 totaling $67,569.06, made across 5 branches within 7 days, just under the $10,000 reporting threshold. 

THOUGHT 2: The relevant statutes include 31 CFR 1020.320 for SAR filing requirements and 31 USC 5324 for structuring transactions to evade reporting requirements.

THOUGHT 3: The narrative will detail WHO (Tracy Lewis, CUST_0053), WHAT (7 cash deposits totaling $67,569.06), WHEN (2025-07-25 to 2025-07-31), WHERE (at Branch_University_District, Branch_Southpark_Ave, Branch_Northgate_Mall, Branch_Eastside_Center, and Branch_Westside_Plaza), and WHY (the pattern of cash deposits just under the reporting threshold is consistent with structuring).

ACTION 1: Tracy Lewis (CUST_0053), a nurse, engaged in suspicious activity characterized by structuring. Between 2025-07-25 and 2025-07-31, she mad

### 📊 SAR Narratives for the Real Suspicious Cases

Narratives are generated for every real case the Risk Analyst rated above **Low**. The benign control (`Other`/`Low`) is skipped, which previews the Phase 4 cost saving: compliance generation only runs where a SAR is plausible.

In [18]:
# Generate narratives for the remaining suspicious real cases
real_narratives = {'CUST_0053': (structuring_narrative, compliance_agent.last_validation)}
for customer_id, analysis in real_results.items():
    if customer_id in real_narratives:
        continue
    if analysis.risk_level == 'Low':
        print(f"⏭️  {customer_id}: {analysis.classification}/{analysis.risk_level} - no SAR narrative generated")
        continue
    narrative = compliance_agent.generate_compliance_narrative(real_cases[customer_id], analysis)
    real_narratives[customer_id] = (narrative, compliance_agent.last_validation)

narrative_table = pd.DataFrame([
    {
        'customer': f"{customer_id} {real_cases[customer_id].customer.name}",
        'classification': real_results[customer_id].classification,
        'words': report['word_count'],
        'complete': narrative.completeness_check,
        'missing_elements': ', '.join(k.upper() for k, v in report['elements'].items() if not v) or '-',
        'citations': '; '.join(c.split(' (')[0] for c in narrative.regulatory_citations),
        'reviewer_warnings': len(report['warnings'])
    }
    for customer_id, (narrative, report) in real_narratives.items()
])
print("\n📊 Real-Case SAR Narratives")
print("=" * 70)
print(narrative_table.to_string(index=False))

for customer_id, (narrative, report) in real_narratives.items():
    print(f"\n🔸 {customer_id} - {real_results[customer_id].classification} ({report['word_count']} words)")
    print(f"   {narrative.narrative}")
    for warning in report['warnings']:
        print(f"   ⚠️  {warning}")

⏭️  CUST_0002: Other/Low - no SAR narrative generated

📊 Real-Case SAR Narratives
                  customer   classification  words  complete missing_elements                    citations  reviewer_warnings
     CUST_0053 Tracy Lewis      Structuring     61      True                - 31 CFR 1020.320; 31 USC 5324                  0
 CUST_0062 Michael Stanley Money_Laundering     74      True                - 31 CFR 1020.320; 18 USC 1956                  0
CUST_0118 Patrick Williams      Structuring     66      True                - 31 CFR 1020.320; 31 USC 5324                  0
   CUST_0125 Kayla Jackson Money_Laundering     67      True                - 31 CFR 1020.320; 18 USC 1956                  0

🔸 CUST_0053 - Structuring (61 words)
   Tracy Lewis (CUST_0053), a nurse, engaged in suspicious activity characterized by structuring. Between 2025-07-25 and 2025-07-31, she made 7 cash deposits totaling $67,569.06, with individual amounts ranging from $9,238.15 to $9,915.61, at Branch_

### 🧪 Synthetic Compliance Scenario Suite

The pre-built `ComplianceOfficerScenarios` give the agent a fixed Risk Analyst finding for each of Structuring, Money_Laundering, Fraud and Sanctions. Each narrative is scored on:
- the 120-word limit;
- the expected typology terms (e.g. *OFAC*, *layering*);
- the expected statutes cited in the text;
- basic structure (dollar amount, subject, activity).

*The suite now builds each narrative on the matching risk scenario's case (previously a case with no transactions, which the schema rejects).*

In [19]:
# Run all pre-built Compliance Officer scenarios
compliance_suite = ComplianceOfficerScenarios()
compliance_summary = compliance_suite.run_all_scenarios(compliance_agent)

compliance_table = pd.DataFrame([
    {
        'scenario': name,
        'words': result['word_count'],
        'within_limit': result['validation']['word_count_valid'],
        'typology_terms': f"{result['validation']['elements_score']:.0%}",
        'statutes_in_text': f"{result['validation']['citations_score']:.0%}",
        'structure': f"{result['validation']['structure_score']:.0%}",
        'missing_terms': ', '.join(result['validation']['elements_missing']) or '-',
        'pass': result['overall_pass']
    }
    for name, result in compliance_summary['detailed_results'].items() if 'validation' in result
])
print("\n" + compliance_table.to_string(index=False))

for name, result in compliance_summary['detailed_results'].items():
    if 'narrative' in result:
        print(f"\n🔸 {name} ({result['word_count']} words):\n   {result['narrative']}")

📝 Running Compliance Officer Scenario Tests...

✅ Testing: Structuring SAR Narrative


   ✅ PASS - Narrative: 70 words
      Preview: Sarah Thompson (CUST_STRUCT_001) conducted three cash deposits totaling $29,200.00 into account ACC_...

✅ Testing: Money Laundering SAR Narrative


   ✅ PASS - Narrative: 69 words
      Preview: International Trading Corp (CUST_ML_001) engaged in suspicious activity involving three wire transfe...

✅ Testing: Fraud SAR Narrative


   ✅ PASS - Narrative: 57 words
      Preview: Elderly Customer (CUST_FRAUD_001) conducted two suspicious transactions from ACC_SAV_2468 between Ja...

✅ Testing: Sanctions Violation SAR Narrative


   ✅ PASS - Narrative: 54 words
      Preview: Import Export LLC (CUST_SANC_001) conducted a wire transfer of $95,000.00 on January 25, 2024, from ...

📊 SUMMARY: 4/4 scenarios passed (100.0%)

                  scenario  words  within_limit typology_terms statutes_in_text structure                                    missing_terms  pass
     structuring_narrative     70          True            50%              50%       67% Bank Secrecy Act, currency transaction reporting  True
money_laundering_narrative     69          True            75%              50%       67%                                         layering  True
           fraud_narrative     57          True           100%              50%      100%                                                -  True
       sanctions_narrative     54          True           100%              50%       67%                                                -  True

🔸 structuring_narrative (70 words):
   Sarah Thompson (CUST_STRUCT_001) conducte

### 🛡️ Error Handling & Compliance Validation

Demonstrated with mocked responses (no API cost), logged to a separate demo file:

| Failure mode | Handling |
|---|---|
| Narrative over 120 words | **ReACT observation loop.** The validator's OBSERVATION (actual word count + guidance) is sent back for revision, up to 3 attempts. |
| Still over the limit after all attempts | `ValueError` (*exceeds 120 word limit*), with a failure audit entry |
| WHERE not named (e.g. "across five locations") | Hard requirement. The OBSERVATION lists the case's branches and account IDs, and the narrative is revised to name them |
| Mandatory SAR citation missing | `31 CFR 1020.320` added as a policy rule, recorded as a normalisation |
| Model claims complete, but WHEN is missing or the SSN appears | `completeness_check` overridden to `False`, and the issues logged, so the human reviewer sees it |
| No JSON / API failure | `ValueError` / `RuntimeError`, with a failure audit entry |

In [20]:
# Demonstrate compliance validation and error handling with mocked LLM responses
import tempfile
from unittest.mock import Mock

def mock_response(content, finish_reason='stop'):
    response = Mock()
    response.choices = [Mock()]
    response.choices[0].message.content = content
    response.choices[0].finish_reason = finish_reason
    response.usage = None
    return response

def narrative_json(narrative, citations=("31 CFR 1020.320 (SAR filing requirement)", "31 USC 5324 (Structuring)"),
                   complete=True):
    return "THOUGHT 1: ...\nACTION 2: OBSERVATION: ...\n```json\n" + json.dumps({
        "narrative": narrative, "narrative_reasoning": "Built from the analyst findings.",
        "regulatory_citations": list(citations), "completeness_check": complete}) + "\n```"

demo_case = real_cases['CUST_0053']
good = ("Tracy Lewis (CUST_0053) made seven cash deposits of $9,238.15 to $9,915.61, totalling $67,569.06, "
        "between 2025-07-25 and 2025-07-31 at the Southpark Ave and Westside Plaza branches. The pattern is "
        "consistent with structuring to avoid the $10,000 Currency Transaction Report threshold under 31 USC 5324.")
unnamed_where = good.replace("at the Southpark Ave and Westside Plaza branches", "across five locations")
overlong = good + " " + " ".join(["Additional detail."] * 60)
no_when = ("Tracy Lewis (CUST_0053) made seven cash deposits totalling $67,569.06 at the Westside Plaza branch, "
           "consistent with structuring.")
with_ssn = good + f" SSN ending {demo_case.customer.ssn_last_4}."

demo_scenarios = {
    'Valid narrative': [mock_response(narrative_json(good))],
    'Over limit -> revised via OBSERVATION': [mock_response(narrative_json(overlong)), mock_response(narrative_json(good))],
    'Over limit on every attempt': [mock_response(narrative_json(overlong))] * 3,
    'Unnamed WHERE -> revised via OBSERVATION': [mock_response(narrative_json(unnamed_where)),
                                                 mock_response(narrative_json(good))],
    'Mandatory citation missing': [mock_response(narrative_json(good, citations=["31 USC 5324 (Structuring)"]))],
    'Claims complete, WHEN missing': [mock_response(narrative_json(no_when))],
    'Claims complete, SSN included': [mock_response(narrative_json(with_ssn))],
    'No JSON on every attempt': [mock_response("I recommend filing a SAR.")] * 3,
    'API connection failure': openai.APIConnectionError(request=Mock()),
}

demo_logger = ExplainabilityLogger(os.path.join(tempfile.mkdtemp(), 'compliance_error_demo.jsonl'))
demo_rows = []
for label, responses in demo_scenarios.items():
    mock_client = Mock()
    mock_client.chat.completions.create.side_effect = responses
    demo_agent = ComplianceOfficerAgent(mock_client, demo_logger, model=MODEL)
    try:
        result = demo_agent.generate_compliance_narrative(demo_case, structuring_result)
        outcome = (f"✅ {len(result.narrative.split())} words, complete={result.completeness_check}, "
                   f"{len(result.regulatory_citations)} citations")
        detail = '; '.join(demo_agent.last_validation['issues']) or 'no issues'
    except (ValueError, RuntimeError) as e:
        outcome = f"🛑 {type(e).__name__}: {str(e).splitlines()[0][:60]}"
        detail = demo_logger.entries[-1]['reasoning']
    demo_rows.append({'scenario': label, 'api_calls': mock_client.chat.completions.create.call_count,
                      'outcome': outcome, 'detail': detail[:60]})
    if label.startswith('Over limit -> revised'):
        observation_sent = mock_client.chat.completions.create.call_args_list[1].kwargs['messages'][-1]['content']

print(pd.DataFrame(demo_rows).to_string(index=False))
print(f"\n🔁 OBSERVATION sent back to the model for revision:\n   {observation_sent[:260]}...")
assert len(demo_logger.entries) == len(demo_scenarios), "Exactly one audit entry per narrative"
print(f"\n✅ {len(demo_logger.entries)} narratives -> {len(demo_logger.entries)} audit entries (one per narrative, success or failure)")

                                scenario  api_calls                                                                    outcome                                                  detail
                         Valid narrative          1                                     ✅ 43 words, complete=True, 2 citations                                               no issues
   Over limit -> revised via OBSERVATION          2                                     ✅ 43 words, complete=True, 2 citations                                               no issues
             Over limit on every attempt          3 🛑 ValueError: Compliance narrative exceeds 120 word limit after 3 attempt(         Word limit validation failed after 3 attempt(s)
Unnamed WHERE -> revised via OBSERVATION          2                                     ✅ 43 words, complete=True, 2 citations                                               no issues
              Mandatory citation missing          1                                  

### 📝 Audit Trail & Cost Review: Both Agents

The notebook's audit trail now holds the full two-agent data flow for each case: `DataLoader` → `RiskAnalyst` → `ComplianceOfficer`. Each entry has its own token usage, so the cost split between the stages can be read straight from the log. That is the basis for the Phase 4 cost-optimisation argument.

In [21]:
# Review the complete audit trail and the cost split between the two agents

with open(audit_log_path) as f:
    audit_entries = [json.loads(line) for line in f]

agent_stats = {}
for entry in audit_entries:
    if entry['agent_type'] == 'DataLoader':
        continue
    details = entry['output_summary']  # stored as a JSON object
    stats = agent_stats.setdefault(entry['agent_type'], {'runs': 0, 'successful': 0, 'api_calls': 0,
                                                         'prompt_tokens': 0, 'completion_tokens': 0, 'latency_ms': 0.0})
    stats['runs'] += 1
    stats['successful'] += entry['success']
    stats['api_calls'] += details['attempts']
    stats['prompt_tokens'] += details['token_usage']['prompt_tokens']
    stats['completion_tokens'] += details['token_usage']['completion_tokens']
    stats['latency_ms'] += entry['execution_time_ms']

# gpt-4o-mini list prices in USD per 1M tokens - update if the model or pricing changes
PRICE_PER_1M = {'prompt_tokens': 0.15, 'completion_tokens': 0.60}
cost_table = pd.DataFrame([
    {
        'agent': agent,
        'runs': s['runs'],
        'successful': s['successful'],
        'api_calls': s['api_calls'],
        'avg_latency_s': round(s['latency_ms'] / s['runs'] / 1000, 1),
        'avg_tokens': round((s['prompt_tokens'] + s['completion_tokens']) / s['runs']),
        'cost_usd': round(sum(s[k] / 1_000_000 * p for k, p in PRICE_PER_1M.items()), 5),
        'cost_per_run_usd': round(sum(s[k] / 1_000_000 * p for k, p in PRICE_PER_1M.items()) / s['runs'], 5)
    }
    for agent, s in agent_stats.items()
])
print("📝 Audit Trail Summary")
print("=" * 70)
print(f"📄 File: {os.path.relpath(audit_log_path, project_root)} ({len(audit_entries)} entries)")
print(cost_table.to_string(index=False))

narrative_entries = [e['output_summary'] for e in audit_entries
                     if e['agent_type'] == 'ComplianceOfficer' and e['success']]
print(f"\n📏 Narrative word counts: {sorted(n['word_count'] for n in narrative_entries)} (limit 120)")
print(f"✅ Narratives passing compliance validation: "
      f"{sum(n['validation']['is_complete'] for n in narrative_entries)} / {len(narrative_entries)}")
print(f"🔁 Narratives needing an OBSERVATION revision: {sum(n['attempts'] > 1 for n in narrative_entries)}")
print(f"⚠️  Narratives with reviewer warnings (accuracy/tone): "
      f"{sum(bool(n['validation']['warnings']) for n in narrative_entries)} / {len(narrative_entries)}")

print("\n🔍 Sample ComplianceOfficer audit entry:")
sample_entry = dict(next(e for e in audit_entries if e['agent_type'] == 'ComplianceOfficer'))
sample_entry['output_summary'] = {k: (v[:300] + '...' if isinstance(v, str) and len(v) > 300 else v) for k, v in sample_entry['output_summary'].items()}  # shorten long text fields for display
print(json.dumps(sample_entry, indent=2))

📝 Audit Trail Summary
📄 File: outputs/audit_logs/agent_development_20261011_185411.jsonl (25 entries)
            agent  runs  successful  api_calls  avg_latency_s  avg_tokens  cost_usd  cost_per_run_usd
      RiskAnalyst    11          11         11            4.1        3103   0.00727           0.00066
ComplianceOfficer     9           9          9            5.9        2657   0.00587           0.00065

📏 Narrative word counts: [54, 57, 61, 66, 67, 68, 69, 70, 74] (limit 120)
✅ Narratives passing compliance validation: 9 / 9
🔁 Narratives needing an OBSERVATION revision: 0
⚠️  Narratives with reviewer warnings (accuracy/tone): 0 / 9

🔍 Sample ComplianceOfficer audit entry:
{
  "timestamp": "2026-10-11T17:55:03.096518+00:00",
  "case_id": "CASE_STRUCTURING_CLASSIC",
  "agent_type": "ComplianceOfficer",
  "action": "generate_narrative",
  "input_summary": {
    "customer_id": "CUST_STRUCT_001",
    "classification": "Structuring",
    "risk_level": "High",
    "confidence_score": 0.9,
 

In [22]:
# COMPLETE AGENT TESTING - full pre-built suite for the foundation and both agents
print("🔬 Complete Agent Testing Workflow")
print("=" * 50)
print("TIER 1: Smoke tests above - live single-case checks of each agent")
print("TIER 2: Pre-built pytest suites - edge cases, parsing, API parameters, prompt structure\n")

full_run = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-q', '-p', 'no:cacheprovider', '--color=no'],
    cwd=project_root, capture_output=True, text=True
)
print(full_run.stdout.strip().splitlines()[-1])
print("🎉 Foundation, Risk Analyst and Compliance Officer all pass - ready for Phase 4 workflow integration!"
      if full_run.returncode == 0 else "⚠️ Fix failing tests before Phase 4")

🔬 Complete Agent Testing Workflow
TIER 1: Smoke tests above - live single-case checks of each agent
TIER 2: Pre-built pytest suites - edge cases, parsing, API parameters, prompt structure



64 passed, 2 warnings in 0.60s
🎉 Foundation, Risk Analyst and Compliance Officer all pass - ready for Phase 4 workflow integration!


## 🔗 Phase 4 Preview: Agent Integration

Once both agents are working, you'll integrate them into a complete workflow.

In [23]:
# TODO: Preview of integrated workflow
# This will be fully implemented in the next notebook

def preview_integrated_workflow():
    """Preview of how agents will work together"""
    
    workflow_steps = [
        "1. 📊 Load and validate case data",
        "2. 🔍 Risk Analyst performs Chain-of-Thought analysis",
        "3. 👤 Human review and approval gate",
        "4. ✅ Compliance Officer generates ReACT narrative (if approved)",
        "5. 📄 Generate complete SAR document",
        "6. 📊 Log audit trail and efficiency metrics"
    ]
    
    print("🔗 Integrated SAR Processing Workflow:")
    for step in workflow_steps:
        print(step)
    
    print("\n💡 Key Benefits:")
    print("• Two-stage processing reduces AI costs")
    print("• Human oversight ensures regulatory compliance")
    print("• Complete audit trails for examination")
    print("• Standardized analytical approaches")

preview_integrated_workflow()

🔗 Integrated SAR Processing Workflow:
1. 📊 Load and validate case data
2. 🔍 Risk Analyst performs Chain-of-Thought analysis
3. 👤 Human review and approval gate
4. ✅ Compliance Officer generates ReACT narrative (if approved)
5. 📄 Generate complete SAR document
6. 📊 Log audit trail and efficiency metrics

💡 Key Benefits:
• Two-stage processing reduces AI costs
• Human oversight ensures regulatory compliance
• Complete audit trails for examination
• Standardized analytical approaches


## 📝 Development Checklist - Two-Tier Testing Approach

### ✅ Risk Analyst Agent (Phase 2)
- [x] Implement Chain-of-Thought system prompt
- [x] Create `analyze_case` method with error handling
- [x] Add JSON parsing and validation
- [x] **TIER 1**: Write simple smoke test (verify basic functionality)
- [x] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [x] Fix any issues identified by test suite

### ✅ Compliance Officer Agent (Phase 3)  
- [x] Implement ReACT system prompt
- [x] Create `generate_compliance_narrative` method
- [x] Add narrative validation (word count, terminology)
- [x] **TIER 1**: Write simple smoke test (verify basic functionality)
- [x] **TIER 2**: Run comprehensive pre-built test suite (10 comprehensive tests)
- [x] Fix any issues identified by test suite

### ✅ Testing Strategy Benefits
- [ ] **Time Savings**: Focus on implementation, not complex test creation
- [ ] **Better Coverage**: Pre-built test suites test edge cases you might miss
- [ ] **Quick Feedback**: Simple smoke tests for rapid development cycles
- [ ] **Professional Validation**: Comprehensive test suites ensure production readiness
- [ ] **Regulatory Compliance**: Built-in checks for SAR requirements

### 💡 **Testing Workflow**
1. **Start with Tier 1**: Write simple smoke tests to verify your agents don't crash
2. **Fix basic issues**: Iterate quickly with simple tests during development
3. **Move to Tier 2**: Run comprehensive test suites when basic functionality works
4. **Analyze results**: Use detailed feedback to improve agent performance
5. **Iterate**: Refine prompts and logic based on test results

## 🚀 Next Steps

1. **Complete Agent Implementation**: Finish both agent classes in the src/ directory
2. **Run Two-Tier Testing**: Start with smoke tests, then comprehensive test suites
3. **Workflow Integration**: Move to the next notebook for complete system integration
4. **Human-in-the-Loop**: Implement decision gates and review processes

## 📊 Available Test Suites Summary

**Risk Analyst Test Suite (10 tests):**
- Agent initialization and configuration
- Case analysis with valid JSON responses
- JSON parsing and error handling
- System prompt structure validation
- API call parameter verification
- Helper method functionality
- Edge case handling

**Compliance Officer Test Suite (10 tests):**
- Agent initialization and configuration
- Narrative generation with valid responses
- Word count validation (≤120 words)
- Regulatory citations inclusion
- JSON parsing and error handling
- ReACT prompt structure validation
- API call parameter verification

**Ready to build intelligent agents with professional-grade testing! 🕵️‍♀️**